# 파노라마 합성 및 객체 검출

> **실험 목표**: SIFT, ORB, ALIKED의 객체 검출 정확도와 실행 시간을 비교하고, 기준 물체의 회전·크기·조명 변화에 대한 강건성을 평가합니다.

> **검출기 변경**: 과제 안내의 AKAZE는 현재 사용 중인 OpenCV 5.0 환경에서 사용할 수 없어 ALIKED로 대체했습니다. ALIKED는 딥러닝 기반 특징점과 LightGlue 매칭을 사용하며, 본 보고서에서는 SIFT·ORB와 함께 성능을 비교합니다.

## 1. 실험 개요

연속 사진을 합성한 파노라마에서 기준 물체의 위치를 찾습니다. 각 방법은 특징점 추출·매칭, Lowe Ratio Test 또는 LightGlue 매칭, RANSAC 기반 호모그래피를 거쳐 검출 결과를 만듭니다.

### 매칭 및 기하 검증 방법

- **Lowe Ratio Test**: SIFT·ORB의 최근접·차근접 매칭 거리를 비교해 모호한 매칭을 제거합니다.
- **LightGlue 매칭**: ALIKED 특징점 사이의 대응 관계를 신경망 기반으로 판단해 신뢰도 높은 매칭을 선택합니다.
- **RANSAC 기반 호모그래피**: 잘못된 매칭(Outlier)을 제외하고, 남은 Inlier로 기준 물체의 평면 좌표 변환과 Polygon 위치를 계산합니다.

## 2. 데이터 구성 및 확장 방식

- 기준 물체: `data/objects/set01/target.jpg`, `set02/target.jpg`, …
- 완성 파노라마: `results/panorama/set01/SIFT/panorama.jpg`, `set01/ORB/panorama.jpg`, `set01/ALIKED/panorama.jpg`
- 파일 번호가 같은 기준 물체와 방법별 파노라마를 자동 연결합니다.
- 각 방법은 자신이 합성한 파노라마에서 객체 검출까지 수행하므로 전체 파이프라인을 비교합니다.
- 현재는 1세트여도, 이후 5세트를 추가하면 코드 수정 없이 전체 케이스에 같은 실험을 적용합니다.

## 3. 평가 기준

- **검출 성공률**: 전체 케이스 중 객체 위치 검출에 성공한 비율
- **평균 탐색 시간**: 특징점 추출과 Coarse-to-Fine 타일 탐색에 걸린 시간
- **매칭 신뢰도**: RANSAC Inlier 수와 Inlier 비율
- **변형 조건**: 기준 상태, 회전 ±30°, 크기 75%·125%, 밝기 70%·130%


### 모듈별 역할

- `src/stitcher.py`: 연속 사진의 특징점 매칭, RANSAC, 호모그래피와 블렌딩으로 파노라마를 합성합니다.
- `src/object_finder.py`: 기준 물체 특징점을 재사용해 Coarse-to-Fine 타일 탐색을 수행하고, 최종 호모그래피를 파노라마 좌표의 Polygon으로 표시합니다.
- `src/pipeline.py`: 합성·검출 모듈을 순서대로 실행하고 이미지·요약 JSON을 저장하는 전체 파이프라인 진입점입니다.
- `src/main.py`: 파노라마 합성 또는 객체 검출 기능을 개별로 확인하는 테스트·디버그 도구입니다.
- `src/project_paths.py`: 데이터와 결과물의 공통 경로, 객체 검출 결과 폴더 규칙을 관리합니다.


In [ ]:
# ==========================================
# 라이브러리 및 프로젝트 모듈 Import
# - 노트북에서 사용하는 공통 설정과 실행 함수를 한곳에서 준비
# ==========================================
import importlib
import logging
import cv2
import matplotlib.pyplot as plt


import src.experiment_runner as experiment_runner
import src.result_cleanup as result_cleanup
import src.settings as settings
import src.stitcher as stitcher
import src.stitcher_create as stitcher_create

importlib.reload(settings)
importlib.reload(stitcher)
importlib.reload(stitcher_create)
importlib.reload(experiment_runner)
importlib.reload(result_cleanup)

from src.experiment_runner import (
    collect_panorama_summary,
    create_detection_matching_summary_figure,
    create_detection_qualitative_figure,
    create_detection_summary_chart_figure,
    create_detection_summary_table_figure,
    create_builtin_stitcher_result_figure,
    create_panorama_filtered_match_figure,
    create_panorama_matching_gallery_figure,
    create_panorama_result_comparison_figure,
    create_panorama_summary_figure,
    discover_test_cases,
    get_experiment_settings,
    run_batch,
    run_panorama_experiments,
    save_experiment_csv,
    summarize_records,
)
from src.logger import logger
from src.project_paths import (
    OBJECT_DIR,
    PANORAMA_RESULT_DIR,
    PANORAMA_SOURCE_DIR,
)
from src.result_cleanup import clear_results
from src.stitcher_create import run_experiments as run_builtin_stitcher_experiments

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 노트북 로그 설정
# - 화면과 results/logs/opencv.log에는 ERROR 이상만 기록
# ==========================================
NOTEBOOK_LOG_LEVEL = logging.ERROR
logger.setLevel(NOTEBOOK_LOG_LEVEL)
logger.propagate = False
for handler in logger.handlers:
    handler.setLevel(NOTEBOOK_LOG_LEVEL)

cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

EXPERIMENT = get_experiment_settings()

In [ ]:
# ==========================================
# 이전 결과 정리
# - opencv.log와 .gitkeep은 보존
# - False면 삭제 대상만 미리 확인
# ==========================================
DELETE_PREVIOUS_RESULTS = True

cleanup_targets = clear_results(dry_run=not DELETE_PREVIOUS_RESULTS)
print('정리 대상 수:', len(cleanup_targets))
if DELETE_PREVIOUS_RESULTS:
    print('이전 결과 삭제 완료')
else:
    print('미리보기 모드입니다. 삭제하려면 True로 변경하세요.')

## 4. 파노라마 합성 실험

각 `data/setNN` 연속 사진 세트에 SIFT, ORB, ALIKED를 적용해 직접 파노라마를 합성한다.
동일한 세트에 OpenCV 내장 Stitcher도 적용해 기준 결과를 생성하고, 방법별 매칭 품질과 합성 결과를 비교한다.

In [ ]:
# ==========================================
# 파노라마 합성 일괄 실행
# - 세트 탐색과 방법별 합성은 experiment_runner.py에서 처리
# ==========================================
panorama_results = run_panorama_experiments(
    source_root=PANORAMA_SOURCE_DIR,
    methods=EXPERIMENT.methods,
    max_side=EXPERIMENT.panorama_max_side,
)

print(f'파노라마 합성 실행 완료: {len(panorama_results)}회')

### 4.1 방법별 파노라마 합성 매칭 통계

인접 사진 쌍의 원시 매칭 수, 필터링 후 매칭 수, RANSAC Inlier 수와 Inlier 비율을 비교한다.

In [ ]:
# ==========================================
# 파노라마 합성 통계 수집 및 비교
# - 세트·방법별 matching_stats.csv 자동 탐색
# ==========================================
panorama_summary_rows = collect_panorama_summary(
    panorama_result_dir=PANORAMA_RESULT_DIR,
    methods=EXPERIMENT.methods,
)
create_panorama_summary_figure(panorama_summary_rows, plt)
plt.show()

In [ ]:
# ==========================================
# 방법별 인접 사진 필터링 매칭 결과
# - 첫 번째 사진 쌍(01.jpg <-> 02.jpg)의 필터링 결과를 비교
# ==========================================
create_panorama_filtered_match_figure(
    summary_rows=panorama_summary_rows,
    panorama_result_dir=PANORAMA_RESULT_DIR,
    pyplot=plt,
)
plt.show()

In [ ]:
# ==========================================
# 파노라마 매칭 이미지 전체 확인
# - raw / filtered / ransac 단계별 이미지를 모두 표시
# ==========================================
MATCHING_VISUAL_SET = 'set01'
MATCHING_VISUAL_METHOD = 'ALIKED'

create_panorama_matching_gallery_figure(
    panorama_result_dir=PANORAMA_RESULT_DIR,
    set_name=MATCHING_VISUAL_SET,
    method=MATCHING_VISUAL_METHOD,
    pyplot=plt,
)
plt.show()

In [ ]:
# ==========================================
# 방법별 파노라마 합성 결과 시각 비교
# - 파노라마와 합성 윤곽선 이미지를 방법별로 표시
# ==========================================
create_panorama_result_comparison_figure(
    summary_rows=panorama_summary_rows,
    pyplot=plt,
)
plt.show()

### 4.2 OpenCV 내장 Stitcher 비교

`cv2.Stitcher_create()`로 동일한 연속 사진 세트를 합성한다. 결과는 `results/panorama/create/세트명/panorama.jpg`에 저장하며, 직접 구현한 방법별 결과와 비교 기준으로 사용한다.

In [ ]:
# ==========================================
# OpenCV 내장 Stitcher 파노라마 생성
# - data/setNN 세트를 자동 탐색
# - 결과는 results/panorama/create/세트명/에 저장
# ==========================================
builtin_stitcher_results = run_builtin_stitcher_experiments(
    max_side=EXPERIMENT.panorama_max_side,
)

print(f'OpenCV 내장 Stitcher 실행 완료: {len(builtin_stitcher_results)}회')

In [ ]:
# ==========================================
# OpenCV 내장 Stitcher 결과 이미지 출력
# ==========================================
create_builtin_stitcher_result_figure(
    result_rows=builtin_stitcher_results,
    pyplot=plt,
)
plt.show()

## 5. 객체 검출 탐색 설정

타일 크기, 탐색 간격, 후보 수, 조기 종료 기준은 `config.json`의 객체 검출 설정을 사용한다.
기준 물체에는 기본 조건과 회전·크기·조명 변화 6개 조건을 적용한다.

In [ ]:
# ==========================================
# 객체 검출 공통 환경 확인
# ==========================================
print('방법:', ', '.join(EXPERIMENT.methods))
print(
    '타일:', EXPERIMENT.tile_size,
    '| Coarse:', EXPERIMENT.coarse_stride,
    '| Fine:', EXPERIMENT.fine_stride,
)
print('실험 조건 수:', len(EXPERIMENT.conditions))

## 6. 객체 검출 테스트 세트 자동 탐색

`data/objects/setN/target.jpg`와 `results/panorama/setN/방법/panorama.jpg`를 세트 번호로 연결한다.
파노라마 세트와 기준 물체가 추가되면 같은 방식으로 자동 확장된다.

In [ ]:
# ==========================================
# 테스트 데이터 자동 탐색
# - setN/target.jpg와 setN/panorama.jpg를 세트 번호로 연결
# ==========================================
TEST_CASES = discover_test_cases(
    target_dir=OBJECT_DIR,
    panorama_result_dir=PANORAMA_RESULT_DIR,
    methods=EXPERIMENT.methods,
)

print('자동 탐색된 테스트 세트 수:', len(TEST_CASES))
print(
    '총 실험 횟수:',
    len(TEST_CASES) * len(EXPERIMENT.methods) * len(EXPERIMENT.conditions),
)

## 7. 객체 검출 일괄 실행

각 테스트 세트와 변형 조건에서 SIFT, ORB, ALIKED를 실행한다. 기본 조건의 검출 결과와
최종 Fine Tile 매칭 이미지는 저장하고, 전체 결과는 CSV로 저장한다.

In [ ]:
# ==========================================
# 객체 검출 일괄 실행 및 CSV 저장
# ==========================================
batch_records = run_batch(
    test_cases=TEST_CASES,
    methods=EXPERIMENT.methods,
    conditions=EXPERIMENT.conditions,
    tile_size=EXPERIMENT.tile_size,
    coarse_stride=EXPERIMENT.coarse_stride,
    fine_stride=EXPERIMENT.fine_stride,
    coarse_top_k=EXPERIMENT.coarse_top_k,
    early_stop_inliers=EXPERIMENT.early_stop_inliers,
    result_root=EXPERIMENT.result_root,
    matching_root=EXPERIMENT.matching_root,
)

summary_rows = summarize_records(batch_records)
csv_paths = save_experiment_csv(
    records=batch_records,
    summary=summary_rows,
    output_dir=EXPERIMENT.metrics_dir,
)

print('실행 완료:', len(batch_records), '회')
print('원시 결과 CSV:', csv_paths['records'])
print('요약 통계 CSV:', csv_paths['summary'])

## 8. 객체 검출 정량적 결과 비교

방법·변형 조건별 성공률, 평균 탐색 시간, 평균 Inlier 비율을 비교한다.

In [ ]:
create_detection_summary_table_figure(
    summary_rows=summary_rows,
    experiment=EXPERIMENT,
    pyplot=plt,
)
plt.show()

### 8.1 방법별 객체 검출 매칭 통계

기본 조건에서 최종 선택된 Fine Tile을 기준으로 방법별 매칭 품질을 비교한다.
`평균 RANSAC 전 매칭`은 SIFT·ORB의 Ratio Test 후, ALIKED의 LightGlue 매칭 후의 수이며,
`평균 Inlier`와 `평균 Inlier 비율`은 검출 성공 케이스만 평균낸 값이다.

In [ ]:
# ==========================================
# 방법별 객체 검출 최종 Tile 매칭 통계
# - 기본 조건의 검출 성공 케이스를 기준으로 평균 매칭 품질 계산
# ==========================================
create_detection_matching_summary_figure(
    records=batch_records,
    experiment=EXPERIMENT,
    pyplot=plt,
    visual_condition='baseline',
)
plt.show()

### 8.2 조건별 성공률과 처리 시간

회전·크기·조명 변화가 검출 성공률과 평균 실행 시간에 미치는 영향을 방법별 막대그래프로 비교한다.

In [ ]:
create_detection_summary_chart_figure(
    summary_rows=summary_rows,
    experiment=EXPERIMENT,
    pyplot=plt,
)
plt.show()

## 9. 객체 검출 정성적 결과 확인

위 행은 파노라마에서 검출된 물체의 Global Polygon이고, 아래 행은 같은 방법에서 선택된
최종 Fine Tile의 Keypoint Matches이다. 수치 결과와 함께 Polygon 위치와 매칭 분포를 확인한다.

In [ ]:
# ==========================================
# 기본 조건 검출 결과 및 최종 Tile 매칭 시각화
# ==========================================
VISUAL_CONDITION = 'baseline'

create_detection_qualitative_figure(
    test_cases=TEST_CASES,
    experiment=EXPERIMENT,
    pyplot=plt,
    visual_condition=VISUAL_CONDITION,
)
plt.show()

## 10. 결론 작성 기준

1. `success_rate`로 검출 안정성과 변화 조건별 취약점을 비교한다.
2. `mean_elapsed`로 처리 속도와 실시간 적용 가능성을 해석한다.
3. 최종 Tile 매칭 통계와 Keypoint Matches 이미지로 기하학적 신뢰도를 확인한다.
4. 정확도·속도·강건성의 균형을 근거로 프로젝트 목적에 적합한 방법을 제안한다.